# 06. 단계적 공개와 설계 비교

단계적 공개는 먼저 짧은 목록을 보고, 필요한 문서 ID를 고른 뒤, 해당 원문만 모델에 전달하는 방식입니다.

## 실행 환경 준비

첫 코드 셀은 노트북에서 사용할 패키지 버전을 맞춥니다. 설치가 끝나면 이어지는 셀을 순서대로 실행합니다.

In [1]:
%pip install -q "openai==3.8.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

In [2]:
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
import os

env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
client = OpenAI()
MODEL = "gpt-5.4-mini"
print("API 실습 준비 완료")

API 실습 준비 완료


## 2. 목록과 원문을 분리합니다

목록은 선택을 돕고, 답은 선택한 원문에서 만듭니다.

In [3]:
documents = [
    {"id": "E01-p2", "title": "이용 시간", "text": "월요일 휴관"},
    {"id": "E01-p3", "title": "대출 규정",
     "text": "7권을 14일 대출. 예약이 없으면 1회 7일 연장."},
    {"id": "E01-p5", "title": "주차 안내", "text": "90분 무료"},
]
catalog = [{"id": d["id"], "title": d["title"]} for d in documents]
print(catalog)

[{'id': 'E01-p2', 'title': '이용 시간'}, {'id': 'E01-p3', 'title': '대출 규정'}, {'id': 'E01-p5', 'title': '주차 안내'}]


## 3. 필요한 ID의 원문만 공개합니다

이 예제에서는 학생이 `E01-p3`을 직접 선택합니다.

In [4]:
selected_id = "E01-p3"
selected = next(d for d in documents if d["id"] == selected_id)
full = "\n".join(d["text"] for d in documents)
compressed = "7권, 14일, 예약 없으면 1회 7일 연장 [E01 p.3]"
print("full 문자 수:", len(full))
print("selected 문자 수:", len(selected["text"]))
print("compressed 문자 수:", len(compressed))

full 문자 수: 43
selected 문자 수: 29
compressed 문자 수: 34


## 4. 선택한 원문으로 답을 생성합니다

In [5]:
prompt = f'근거 [E01 p.3]: {selected["text"]}\n질문: 대출과 연장 조건은?'

In [6]:
response = client.responses.create(
    model=MODEL,
    instructions="근거의 조건과 출처를 빠뜨리지 마세요.",
    input=prompt,
    max_output_tokens=300,
    reasoning={"effort": "none"},
)
print(response.output_text)
print("input_tokens:", response.usage.input_tokens)
print("output_tokens:", response.usage.output_tokens)

근거 [E01 p.3]에 따르면 대출과 연장 조건은 다음과 같습니다.

- **대출 기간**: **7권을 14일 대출**
- **연장 조건**: **예약이 없으면 1회 7일 연장 가능**

즉, **14일 대출**이 기본이고, **해당 도서에 예약이 없을 경우 1회에 한해 7일 연장**할 수 있습니다.
input_tokens: 67
output_tokens: 105


## 5. 일부러 실패해 보기

질문에 필요한 문서를 선택하지 않으면 토큰은 줄어도 답의 근거가 사라집니다.

In [7]:
def require_document(selected_id: str, required_id: str) -> None:
    if selected_id != required_id:
        raise ValueError(f"필요한 문서 {required_id}가 선택되지 않았습니다.")

try:
    require_document("E01-p5", "E01-p3")
except ValueError as error:
    print("예상한 실패:", error)
else:
    raise AssertionError("문서 선택 오류가 발생해야 합니다.")

예상한 실패: 필요한 문서 E01-p3가 선택되지 않았습니다.


## 정리

목록, 선택, 원문 읽기를 순서대로 수행합니다. 토큰 감소와 필요한 조건의 보존 여부를 함께 비교합니다.